In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
%run "../00.common/02.bronze-helpers.py"

In [0]:
# COMMAND ----------
 
# MAGIC %run "../00_common/01_environment_config"
 
 
# COMMAND ----------
 
# MAGIC %run "../00_common/02_bronze_helpers"
 
 
# COMMAND ----------
 
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)
 
 
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
source_file = f"{s3_root_path}/landing/constructors.json"
 
target_path = f"{s3_root_path}/Bronze/Nisarga/constructors"
 
print("Source :", source_file)
print("Target :", target_path)
 
 
# COMMAND ----------
 
# ============================================================
# DEFINE SCHEMA
# ============================================================
 
constructors_schema = StructType([
    StructField("constructorId", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])
 
 
# COMMAND ----------
 
# ============================================================
# READ CONSTRUCTORS JSON
# ============================================================
 
constructors_df = (
    spark.read
        .schema(constructors_schema)
        .json(source_file)
)
 
display(constructors_df.limit(20))
 
 
# COMMAND ----------
 
# ============================================================
# ADD INGESTION METADATA
# ============================================================
 
constructors_bronze_df = add_ingestion_metadata(constructors_df)
 
display(constructors_bronze_df.limit(20))
 
 
# COMMAND ----------
 
# ============================================================
# WRITE TO BRONZE
# ============================================================
 
(
    constructors_bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(target_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# VALIDATE BRONZE DATA
# ============================================================
 
bronze_constructors_df = (
    spark.read
        .format("delta")
        .load(target_path)
)
 
display(bronze_constructors_df.limit(20))
 
bronze_constructors_df.printSchema()
 
print("Bronze Constructors Records :", bronze_constructors_df.count())